# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_04 — Implied Volatility Root Finding

### Research question

How can we recover implied volatility from option market prices robustly, while detecting bad quotes, no-arbitrage violations, low-Vega instability, and solver failures?

This notebook builds a production-style implied volatility solver around the Black-Scholes-Merton formula.

It covers:

1. implied volatility as a root-finding problem;
2. option price bounds and quote validity;
3. BSM Vega and monotonicity in volatility;
4. Newton-Raphson inversion;
5. bisection fallback;
6. optional SciPy/Brent-style fallback;
7. solver diagnostics and status codes;
8. synthetic option-chain generation;
9. bid/ask noise and invalid quote injection;
10. batch implied-volatility extraction;
11. smile and surface reconstruction;
12. no-arbitrage checks across strikes;
13. low-Vega and extreme-moneyness failure cases;
14. saved IV surface and solver audit report.

The central idea is:

> Implied volatility is not an observed input. It is the volatility parameter that makes a pricing model reproduce a market option price.

## 1. Definition of implied volatility

Let $C^{\text{mkt}}$ be the observed market price of a European call.

The Black-Scholes-Merton call price is:

$$
C^{BSM}(S_0,K,T,r,q,\sigma)
$$
The implied volatility is the value $\sigma_{\text{imp}}$ satisfying:

$$
\begin{aligned}
C^{BSM}(S_0,K,T,r,q,\sigma_{\text{imp}}) &= C^{\text{mkt}}
\end{aligned}
$$
Equivalently, solve the root-finding problem:

$$
\begin{aligned}
f(\sigma) &= C^{BSM}(\sigma) - C^{\text{mkt}} \\
&= 0
\end{aligned}
$$
The same idea applies to puts.

Implied volatility is model-dependent. It is the volatility that makes the BSM model match the observed option price. It is not necessarily the realised future volatility of the underlying.

## 2. Price bounds before root finding

Before trying to invert an option price, check no-arbitrage bounds.

For a European call with continuous dividend yield $q$:

$$
\begin{aligned}
\max(S_0e^{-qT} - Ke^{-rT},0) &\leq C \\
&\leq S_0e^{-qT}
\end{aligned}
$$
For a European put:

$$
\begin{aligned}
\max(Ke^{-rT} - S_0e^{-qT},0) &\leq P \\
&\leq Ke^{-rT}
\end{aligned}
$$
If the market price is outside these bounds, there may be:

- a bad quote;
- stale data;
- wrong interest rate or dividend input;
- American exercise premium;
- missing corporate action adjustment;
- bid/ask midpoint distortion;
- wrong option type or contract specification.

A robust solver should reject or flag prices outside the theoretical bounds instead of returning a meaningless volatility.

## 3. Vega and monotonicity

BSM Vega is the derivative of option price with respect to volatility:

$$
\begin{aligned}
\text{Vega} &= \frac{\partial V}{\partial \sigma} \\
&= S_0e^{-qT}\phi(d_1)\sqrt T
\end{aligned}
$$
where $\phi$ is the standard normal density.

For plain European calls and puts under BSM, Vega is positive for non-degenerate options:

$$
\frac{\partial V}{\partial \sigma} > 0
$$
So the option price is monotonic in volatility.

This is why implied volatility inversion is usually well-posed.

However, Vega can become very small for:

- very short maturities;
- deep in-the-money options;
- deep out-of-the-money options;
- prices very close to intrinsic value.

Low Vega makes Newton-Raphson unstable because the update divides by Vega.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt, pi
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class OptionMarketConfig:
    """
    Synthetic option market configuration.
    """
    spot: float = 100.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.01
    seed: int = 42


config = OptionMarketConfig()
config

## 5. Black-Scholes-Merton pricing functions

We implement the BSM call and put formulas.

For calls:

$$
\begin{aligned}
C &= S_0e^{-qT}N(d_1) \\
&\quad - Ke^{-rT}N(d_2)
\end{aligned}
$$
For puts:

$$
\begin{aligned}
P &= Ke^{-rT}N(-d_2) \\
&\quad - S_0e^{-qT}N(-d_1)
\end{aligned}
$$
where:

$$
\begin{aligned}
d_1 &= \frac{ \log(S_0/K) \\
&\quad + (r-q+\frac{1}{2}\sigma^2)T } {\sigma\sqrt T}
\end{aligned}
$$
$$
d_2 = d_1-\sigma\sqrt T
$$

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF using the error function.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal PDF.
    """
    x = np.asarray(x)
    return np.exp(-0.5 * x ** 2) / np.sqrt(2.0 * np.pi)


def bsm_d1_d2(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> tuple[float, float]:
    """
    Compute BSM d1 and d2.
    """
    if spot <= 0 or strike <= 0:
        raise ValueError("spot and strike must be positive.")

    if maturity <= 0:
        raise ValueError("maturity must be positive.")

    if volatility <= 0:
        raise ValueError("volatility must be positive.")

    d1 = (
        log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * maturity
    ) / (volatility * sqrt(maturity))

    d2 = d1 - volatility * sqrt(maturity)

    return d1, d2


def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> float:
    """
    Black-Scholes-Merton European option price.
    """
    d1, d2 = bsm_d1_d2(
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        return float(
            discounted_spot * normal_cdf(d1)
            - discounted_strike * normal_cdf(d2)
        )

    if option_type == "put":
        return float(
            discounted_strike * normal_cdf(-d2)
            - discounted_spot * normal_cdf(-d1)
        )

    raise ValueError("option_type must be 'call' or 'put'.")


def bsm_vega(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> float:
    """
    BSM Vega: derivative of option price with respect to volatility.
    """
    d1, _ = bsm_d1_d2(
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    return float(
        spot
        * exp(-dividend_yield * maturity)
        * normal_pdf(d1)
        * sqrt(maturity)
    )

In [ ]:
example_price = bsm_price(
    spot=config.spot,
    strike=100.0,
    maturity=1.0,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=0.20,
    option_type="call"
)

example_vega = bsm_vega(
    spot=config.spot,
    strike=100.0,
    maturity=1.0,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=0.20
)

pd.Series({
    "example_call_price": example_price,
    "example_vega": example_vega
})

## 6. Option price bounds

The solver will first check whether the price is inside theoretical bounds.

This is especially important for batch option-chain processing where some quotes may be stale or internally inconsistent.

In [ ]:
def option_price_bounds(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str
) -> tuple[float, float]:
    """
    No-arbitrage lower and upper bounds for European call/put under continuous dividend yield.
    """
    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        lower = max(discounted_spot - discounted_strike, 0.0)
        upper = discounted_spot
    elif option_type == "put":
        lower = max(discounted_strike - discounted_spot, 0.0)
        upper = discounted_strike
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    return lower, upper


def check_price_bounds(
    market_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    tolerance: float = 1e-10
) -> dict:
    """
    Check whether option price lies inside no-arbitrage bounds.
    """
    lower, upper = option_price_bounds(
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type
    )

    passed = (market_price >= lower - tolerance) and (market_price <= upper + tolerance)

    return {
        "lower_bound": lower,
        "upper_bound": upper,
        "within_bounds": passed,
        "distance_to_lower": market_price - lower,
        "distance_to_upper": upper - market_price
    }


check_price_bounds(
    market_price=example_price,
    spot=config.spot,
    strike=100.0,
    maturity=1.0,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    option_type="call"
)

## 7. Implied volatility by Newton-Raphson

Newton-Raphson solves:

$$
f(\sigma)=0
$$
using the iteration:

$$
\begin{aligned}
\sigma_{n+1} &= \sigma_n \\
&\quad - \frac{f(\sigma_n)}{f'(\sigma_n)}
\end{aligned}
$$
For implied volatility:

$$
\begin{aligned}
f(\sigma) &= V^{BSM}(\sigma) - V^{mkt}
\end{aligned}
$$
and:

$$
f'(\sigma) = \text{Vega}(\sigma)
$$
So:

$$
\begin{aligned}
\sigma_{n+1} &= \sigma_n \\
&\quad - \frac{ V^{BSM}(\sigma_n)-V^{mkt} } {\text{Vega}(\sigma_n)}
\end{aligned}
$$
Newton is fast when Vega is healthy and the starting guess is good.  
It is fragile when Vega is small or the starting guess is poor.

In [ ]:
@dataclass
class ImpliedVolResult:
    """
    Result object for implied volatility solving.
    """
    implied_vol: float | None
    status: str
    method: str
    iterations: int
    model_price: float | None
    market_price: float
    pricing_error: float | None
    vega: float | None
    lower_bound: float
    upper_bound: float
    message: str


def implied_vol_newton(
    market_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    initial_vol: float = 0.20,
    tolerance: float = 1e-10,
    max_iter: int = 100,
    min_vol: float = 1e-8,
    max_vol: float = 5.0,
    min_vega: float = 1e-8
) -> tuple[ImpliedVolResult, pd.DataFrame]:
    """
    Implied volatility solver using Newton-Raphson.

    Returns the result and an iteration log.
    """
    bounds = check_price_bounds(
        market_price=market_price,
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type
    )

    if not bounds["within_bounds"]:
        result = ImpliedVolResult(
            implied_vol=None,
            status="failed_bounds",
            method="newton",
            iterations=0,
            model_price=None,
            market_price=market_price,
            pricing_error=None,
            vega=None,
            lower_bound=bounds["lower_bound"],
            upper_bound=bounds["upper_bound"],
            message="Market price outside no-arbitrage bounds."
        )
        return result, pd.DataFrame()

    sigma = float(np.clip(initial_vol, min_vol, max_vol))
    rows = []

    for iteration in range(1, max_iter + 1):
        price = bsm_price(
            spot=spot,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            dividend_yield=dividend_yield,
            volatility=sigma,
            option_type=option_type
        )

        vega = bsm_vega(
            spot=spot,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            dividend_yield=dividend_yield,
            volatility=sigma
        )

        error = price - market_price

        rows.append({
            "iteration": iteration,
            "sigma": sigma,
            "model_price": price,
            "market_price": market_price,
            "pricing_error": error,
            "vega": vega
        })

        if abs(error) < tolerance:
            result = ImpliedVolResult(
                implied_vol=sigma,
                status="converged",
                method="newton",
                iterations=iteration,
                model_price=price,
                market_price=market_price,
                pricing_error=error,
                vega=vega,
                lower_bound=bounds["lower_bound"],
                upper_bound=bounds["upper_bound"],
                message="Newton converged."
            )
            return result, pd.DataFrame(rows)

        if vega < min_vega:
            break

        sigma_next = sigma - error / vega

        if (not np.isfinite(sigma_next)) or sigma_next <= min_vol or sigma_next >= max_vol:
            break

        sigma = float(sigma_next)

    final_row = rows[-1] if rows else {}

    result = ImpliedVolResult(
        implied_vol=None,
        status="failed_newton",
        method="newton",
        iterations=len(rows),
        model_price=final_row.get("model_price"),
        market_price=market_price,
        pricing_error=final_row.get("pricing_error"),
        vega=final_row.get("vega"),
        lower_bound=bounds["lower_bound"],
        upper_bound=bounds["upper_bound"],
        message="Newton failed or became unstable."
    )

    return result, pd.DataFrame(rows)

In [ ]:
true_vol = 0.23
market_price = bsm_price(
    spot=config.spot,
    strike=105.0,
    maturity=0.75,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=true_vol,
    option_type="call"
)

newton_result, newton_log = implied_vol_newton(
    market_price=market_price,
    spot=config.spot,
    strike=105.0,
    maturity=0.75,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    option_type="call",
    initial_vol=0.20
)

asdict(newton_result)

In [ ]:
newton_log

## 8. Newton convergence plot

The convergence path is useful for diagnosing solver behaviour.

For ordinary at-the-money options, Newton usually converges quickly.

For short-dated or deep out-of-the-money options, the path can be unstable because Vega is small.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(newton_log["iteration"], newton_log["sigma"], marker="o")
plt.axhline(true_vol, linestyle="--", label="True volatility")
plt.title("Newton-Raphson Implied Volatility Iterations")
plt.xlabel("Iteration")
plt.ylabel("Volatility")
plt.legend()
plt.show()

## 9. Bisection fallback

Bisection is slower than Newton but more robust.

If $f(a)$ and $f(b)$ have opposite signs on a bracket $[a,b]$, then a continuous function has a root in the interval.

For implied volatility, monotonicity means a wide bracket such as:

$$
[10^{-8}, 5]
$$
often works if the option price is inside no-arbitrage bounds.

Bisection does not require Vega.

In [ ]:
def implied_vol_bisection(
    market_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    tolerance: float = 1e-10,
    max_iter: int = 200,
    min_vol: float = 1e-8,
    max_vol: float = 5.0
) -> tuple[ImpliedVolResult, pd.DataFrame]:
    """
    Implied volatility solver using bisection.
    """
    bounds = check_price_bounds(
        market_price=market_price,
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type
    )

    if not bounds["within_bounds"]:
        result = ImpliedVolResult(
            implied_vol=None,
            status="failed_bounds",
            method="bisection",
            iterations=0,
            model_price=None,
            market_price=market_price,
            pricing_error=None,
            vega=None,
            lower_bound=bounds["lower_bound"],
            upper_bound=bounds["upper_bound"],
            message="Market price outside no-arbitrage bounds."
        )
        return result, pd.DataFrame()

    def objective(vol: float) -> float:
        return (
            bsm_price(
                spot=spot,
                strike=strike,
                maturity=maturity,
                risk_free_rate=risk_free_rate,
                dividend_yield=dividend_yield,
                volatility=vol,
                option_type=option_type
            )
            - market_price
        )

    low = min_vol
    high = max_vol
    f_low = objective(low)
    f_high = objective(high)

    rows = []

    if f_low * f_high > 0:
        result = ImpliedVolResult(
            implied_vol=None,
            status="failed_bracket",
            method="bisection",
            iterations=0,
            model_price=None,
            market_price=market_price,
            pricing_error=None,
            vega=None,
            lower_bound=bounds["lower_bound"],
            upper_bound=bounds["upper_bound"],
            message="Bisection bracket does not contain a sign change."
        )
        return result, pd.DataFrame()

    mid = np.nan
    price_mid = np.nan
    f_mid = np.nan
    vega_mid = np.nan

    for iteration in range(1, max_iter + 1):
        mid = 0.5 * (low + high)
        price_mid = bsm_price(
            spot=spot,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            dividend_yield=dividend_yield,
            volatility=mid,
            option_type=option_type
        )
        f_mid = price_mid - market_price

        vega_mid = bsm_vega(
            spot=spot,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            dividend_yield=dividend_yield,
            volatility=mid
        )

        rows.append({
            "iteration": iteration,
            "low": low,
            "high": high,
            "mid": mid,
            "model_price": price_mid,
            "pricing_error": f_mid,
            "vega": vega_mid
        })

        if abs(f_mid) < tolerance or (high - low) < tolerance:
            result = ImpliedVolResult(
                implied_vol=mid,
                status="converged",
                method="bisection",
                iterations=iteration,
                model_price=price_mid,
                market_price=market_price,
                pricing_error=f_mid,
                vega=vega_mid,
                lower_bound=bounds["lower_bound"],
                upper_bound=bounds["upper_bound"],
                message="Bisection converged."
            )
            return result, pd.DataFrame(rows)

        if f_low * f_mid <= 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    result = ImpliedVolResult(
        implied_vol=mid,
        status="failed_max_iter",
        method="bisection",
        iterations=max_iter,
        model_price=price_mid,
        market_price=market_price,
        pricing_error=f_mid,
        vega=vega_mid,
        lower_bound=bounds["lower_bound"],
        upper_bound=bounds["upper_bound"],
        message="Bisection hit max_iter."
    )

    return result, pd.DataFrame(rows)

In [ ]:
bisection_result, bisection_log = implied_vol_bisection(
    market_price=market_price,
    spot=config.spot,
    strike=105.0,
    maturity=0.75,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    option_type="call"
)

asdict(bisection_result)

## 10. Robust solver: Newton first, bisection fallback

A practical implied-volatility solver should combine speed and reliability.

Workflow:

1. check price bounds;
2. attempt Newton-Raphson;
3. if Newton converges, return;
4. otherwise use bisection fallback;
5. return full diagnostics.

This is the pattern used in many production solvers: a fast method first, a robust method second.

In [ ]:
def implied_vol_robust(
    market_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    initial_vol: float = 0.20
) -> tuple[ImpliedVolResult, pd.DataFrame]:
    """
    Robust implied volatility solver.

    Tries Newton first. If it fails, falls back to bisection.
    """
    newton_result, newton_log = implied_vol_newton(
        market_price=market_price,
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type,
        initial_vol=initial_vol
    )

    if newton_result.status == "converged":
        newton_log = newton_log.copy()
        newton_log["stage"] = "newton"
        return newton_result, newton_log

    bisection_result, bisection_log = implied_vol_bisection(
        market_price=market_price,
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type
    )

    bisection_log = bisection_log.copy()
    bisection_log["stage"] = "bisection"

    if bisection_result.status == "converged":
        bisection_result.method = "newton_then_bisection"
        bisection_result.message = "Newton failed; bisection fallback converged."

    combined_log = pd.concat(
        [
            newton_log.assign(stage="newton"),
            bisection_log
        ],
        ignore_index=True
    )

    return bisection_result, combined_log


robust_result, robust_log = implied_vol_robust(
    market_price=market_price,
    spot=config.spot,
    strike=105.0,
    maturity=0.75,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    option_type="call",
    initial_vol=0.20
)

asdict(robust_result)

## 11. Synthetic implied volatility surface

We generate a synthetic volatility smile/surface.

Let log-moneyness be:

$$
k = \log(K/S_0)
$$
A simple synthetic implied volatility surface can be:

$$
\begin{aligned}
\sigma_{\text{true}}(k,T) &= \sigma_0 \\
&\quad + a k^2 \\
&\quad + b k \\
&\quad + c e^{-T}
\end{aligned}
$$
This creates:

- smile curvature;
- skew;
- term-structure variation.

We then generate option prices from the surface and add bid/ask noise.

In [ ]:
def synthetic_true_implied_volatility(
    spot: float,
    strike: float,
    maturity: float
) -> float:
    """
    Synthetic volatility surface used to generate option prices.
    """
    k = log(strike / spot)

    vol = (
        0.18
        + 0.35 * k ** 2
        - 0.10 * k
        + 0.04 * exp(-1.5 * maturity)
    )

    return float(np.clip(vol, 0.05, 1.50))


def generate_synthetic_option_chain(
    config: OptionMarketConfig,
    strikes: np.ndarray,
    maturities: np.ndarray,
    option_types: list[str],
    seed: int = 42
) -> pd.DataFrame:
    """
    Generate synthetic option quotes from a known IV surface.
    """
    local_rng = np.random.default_rng(seed)

    rows = []

    for maturity in maturities:
        for strike in strikes:
            true_iv = synthetic_true_implied_volatility(config.spot, strike, maturity)

            for option_type in option_types:
                theoretical_price = bsm_price(
                    spot=config.spot,
                    strike=float(strike),
                    maturity=float(maturity),
                    risk_free_rate=config.risk_free_rate,
                    dividend_yield=config.dividend_yield,
                    volatility=true_iv,
                    option_type=option_type
                )

                vega = bsm_vega(
                    spot=config.spot,
                    strike=float(strike),
                    maturity=float(maturity),
                    risk_free_rate=config.risk_free_rate,
                    dividend_yield=config.dividend_yield,
                    volatility=true_iv
                )

                # Bid/ask spread increases for low-vega options.
                quote_spread = max(0.02, 0.002 * theoretical_price + 0.15 / (1 + vega))

                mid_noise = local_rng.normal(0.0, 0.15 * quote_spread)
                mid_price = max(theoretical_price + mid_noise, 1e-6)

                bid = max(mid_price - quote_spread / 2, 0.0)
                ask = mid_price + quote_spread / 2

                rows.append({
                    "spot": config.spot,
                    "strike": float(strike),
                    "maturity": float(maturity),
                    "risk_free_rate": config.risk_free_rate,
                    "dividend_yield": config.dividend_yield,
                    "option_type": option_type,
                    "true_iv": true_iv,
                    "theoretical_price": theoretical_price,
                    "bid": bid,
                    "ask": ask,
                    "mid": 0.5 * (bid + ask),
                    "quote_spread": ask - bid,
                    "true_vega": vega,
                    "log_moneyness": log(strike / config.spot)
                })

    chain = pd.DataFrame(rows)

    return chain


strikes = np.linspace(60, 140, 41)
maturities = np.array([7, 14, 30, 60, 90, 180, 365]) / 365.0

clean_chain = generate_synthetic_option_chain(
    config=config,
    strikes=strikes,
    maturities=maturities,
    option_types=["call", "put"],
    seed=config.seed
)

clean_chain.head()

## 12. Injecting bad quotes

Real option chains often contain bad or unusable quotes:

- zero bids;
- stale asks;
- negative bid/ask spread;
- prices outside theoretical bounds;
- crossed markets;
- tiny prices with low Vega;
- missing contracts.

We deliberately inject a few bad quotes so the solver can demonstrate failure handling.

In [ ]:
def inject_bad_option_quotes(chain: pd.DataFrame, seed: int = 123) -> pd.DataFrame:
    """
    Inject a few bad quotes into the synthetic option chain.
    """
    local_rng = np.random.default_rng(seed)
    out = chain.copy()
    out["is_bad_quote_injected"] = False
    out["bad_quote_type"] = "none"

    n = len(out)

    # 1. Price above upper bound.
    idx_upper = local_rng.choice(out.index, size=4, replace=False)
    for idx in idx_upper:
        row = out.loc[idx]
        _, upper = option_price_bounds(
            spot=row["spot"],
            strike=row["strike"],
            maturity=row["maturity"],
            risk_free_rate=row["risk_free_rate"],
            dividend_yield=row["dividend_yield"],
            option_type=row["option_type"]
        )
        out.loc[idx, "mid"] = upper * 1.05
        out.loc[idx, "bid"] = upper * 1.04
        out.loc[idx, "ask"] = upper * 1.06
        out.loc[idx, "is_bad_quote_injected"] = True
        out.loc[idx, "bad_quote_type"] = "above_upper_bound"

    # 2. Price below lower bound.
    remaining = out.index.difference(idx_upper)
    idx_lower = local_rng.choice(remaining, size=4, replace=False)
    for idx in idx_lower:
        row = out.loc[idx]
        lower, _ = option_price_bounds(
            spot=row["spot"],
            strike=row["strike"],
            maturity=row["maturity"],
            risk_free_rate=row["risk_free_rate"],
            dividend_yield=row["dividend_yield"],
            option_type=row["option_type"]
        )
        bad_mid = max(lower - 0.05, 0.0)
        out.loc[idx, "mid"] = bad_mid
        out.loc[idx, "bid"] = max(bad_mid - 0.01, 0.0)
        out.loc[idx, "ask"] = bad_mid + 0.01
        out.loc[idx, "is_bad_quote_injected"] = True
        out.loc[idx, "bad_quote_type"] = "below_lower_bound"

    # 3. Crossed bid/ask.
    remaining = out.index.difference(idx_upper).difference(idx_lower)
    idx_crossed = local_rng.choice(remaining, size=4, replace=False)
    for idx in idx_crossed:
        old_bid = out.loc[idx, "bid"]
        old_ask = out.loc[idx, "ask"]
        out.loc[idx, "bid"] = old_ask
        out.loc[idx, "ask"] = old_bid
        out.loc[idx, "mid"] = 0.5 * (old_bid + old_ask)
        out.loc[idx, "is_bad_quote_injected"] = True
        out.loc[idx, "bad_quote_type"] = "crossed_market"

    out["quote_spread"] = out["ask"] - out["bid"]

    return out


option_chain = inject_bad_option_quotes(clean_chain, seed=2026)

option_chain["bad_quote_type"].value_counts()

## 13. Quote validation

Before solving implied volatility, validate each quote.

Checks:

1. bid and ask are non-negative;
2. ask is at least bid;
3. midpoint lies within theoretical bounds;
4. maturity is positive;
5. strike and spot are positive.

Invalid quotes should be flagged and excluded from surface fitting.

In [ ]:
def validate_option_quote(row: pd.Series) -> dict:
    """
    Validate one option quote.
    """
    issues = []

    if row["spot"] <= 0:
        issues.append("non_positive_spot")

    if row["strike"] <= 0:
        issues.append("non_positive_strike")

    if row["maturity"] <= 0:
        issues.append("non_positive_maturity")

    if row["bid"] < 0 or row["ask"] < 0:
        issues.append("negative_bid_or_ask")

    if row["ask"] < row["bid"]:
        issues.append("crossed_market")

    bounds = check_price_bounds(
        market_price=row["mid"],
        spot=row["spot"],
        strike=row["strike"],
        maturity=row["maturity"],
        risk_free_rate=row["risk_free_rate"],
        dividend_yield=row["dividend_yield"],
        option_type=row["option_type"]
    )

    if not bounds["within_bounds"]:
        issues.append("mid_outside_theoretical_bounds")

    return {
        "quote_is_valid": len(issues) == 0,
        "quote_issues": ",".join(issues) if issues else "none",
        **bounds
    }


validation_records = option_chain.apply(validate_option_quote, axis=1, result_type="expand")
option_chain_validated = pd.concat([option_chain.reset_index(drop=True), validation_records], axis=1)

option_chain_validated.head()

In [ ]:
quote_validation_summary = option_chain_validated["quote_issues"].value_counts()

quote_validation_summary

## 14. Batch implied-volatility extraction

We now solve implied volatility for every option quote.

The solver returns:

- implied volatility;
- status;
- method;
- iteration count;
- final pricing error;
- Vega;
- bounds;
- diagnostic message.

This is the key difference between a research demo and a robust data pipeline.

In [ ]:
def solve_chain_implied_vols(chain: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Solve implied volatility for every row in an option chain.

    Returns
    -------
    solved_chain:
        Option chain with solver diagnostics.

    iteration_logs:
        Iteration-level logs for all options.
    """
    solved_rows = []
    logs = []

    for row_id, row in chain.reset_index(drop=True).iterrows():
        result, log_df = implied_vol_robust(
            market_price=float(row["mid"]),
            spot=float(row["spot"]),
            strike=float(row["strike"]),
            maturity=float(row["maturity"]),
            risk_free_rate=float(row["risk_free_rate"]),
            dividend_yield=float(row["dividend_yield"]),
            option_type=str(row["option_type"]),
            initial_vol=0.20
        )

        result_dict = asdict(result)
        result_dict["row_id"] = row_id

        solved_rows.append(result_dict)

        if not log_df.empty:
            log_df = log_df.copy()
            log_df["row_id"] = row_id
            logs.append(log_df)

    solved = pd.DataFrame(solved_rows)

    solved_chain = pd.concat(
        [
            chain.reset_index(drop=True),
            solved.drop(columns=["market_price", "lower_bound", "upper_bound"])
        ],
        axis=1
    )

    iteration_logs = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()

    return solved_chain, iteration_logs


solved_chain, iteration_logs = solve_chain_implied_vols(option_chain_validated)

solved_chain.head()

In [ ]:
solver_status_summary = solved_chain["status"].value_counts()

solver_status_summary

## 15. Accuracy against known synthetic volatility

For valid quotes generated from a known synthetic volatility surface, the recovered IV should be close to the true IV.

The difference reflects bid/ask noise, injected quote errors, and numerical tolerance.

In [ ]:
solved_chain["iv_error"] = solved_chain["implied_vol"] - solved_chain["true_iv"]
solved_chain["abs_iv_error"] = solved_chain["iv_error"].abs()

accuracy_summary = (
    solved_chain[solved_chain["status"] == "converged"]
    .groupby("option_type")
    .agg(
        n=("implied_vol", "count"),
        mean_abs_iv_error=("abs_iv_error", "mean"),
        median_abs_iv_error=("abs_iv_error", "median"),
        max_abs_iv_error=("abs_iv_error", "max"),
        mean_iterations=("iterations", "mean")
    )
    .reset_index()
)

accuracy_summary

In [ ]:
plt.figure(figsize=(10, 6))
valid_solved = solved_chain[solved_chain["status"] == "converged"]
plt.scatter(valid_solved["true_iv"], valid_solved["implied_vol"], alpha=0.7)
min_v = min(valid_solved["true_iv"].min(), valid_solved["implied_vol"].min())
max_v = max(valid_solved["true_iv"].max(), valid_solved["implied_vol"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Recovered Implied Volatility vs True Synthetic Volatility")
plt.xlabel("True IV")
plt.ylabel("Recovered IV")
plt.show()

## 16. Smile reconstruction

We can inspect implied volatility by strike for each maturity.

A volatility smile or skew is a curve:

$$
K \mapsto \sigma_{\text{imp}}(K,T)
$$
for fixed maturity $T$.

The synthetic surface was designed with both curvature and skew.

In [ ]:
def plot_smile_for_maturity(
    solved_chain: pd.DataFrame,
    maturity: float,
    option_type: str = "call"
) -> None:
    """
    Plot true and recovered implied-volatility smile for one maturity.
    """
    part = solved_chain[
        (solved_chain["maturity"] == maturity)
        & (solved_chain["option_type"] == option_type)
        & (solved_chain["status"] == "converged")
    ].sort_values("strike")

    plt.figure(figsize=(10, 6))
    plt.plot(part["strike"], part["true_iv"], marker="o", label="True IV")
    plt.plot(part["strike"], part["implied_vol"], marker="x", label="Recovered IV")
    plt.title(f"Implied Volatility Smile: {option_type}, T={maturity:.3f} years")
    plt.xlabel("Strike")
    plt.ylabel("Implied volatility")
    plt.legend()
    plt.show()


for maturity in [maturities[0], maturities[2], maturities[-1]]:
    plot_smile_for_maturity(solved_chain, maturity=float(maturity), option_type="call")

## 17. Surface reconstruction

A volatility surface is:

$$
(K,T) \mapsto \sigma_{\text{imp}}(K,T)
$$
We visualise the recovered call surface as a heatmap over strike and maturity.

In real markets, this surface should also be checked for static arbitrage, smoothness, and quote quality.

In [ ]:
call_surface = (
    solved_chain[
        (solved_chain["option_type"] == "call")
        & (solved_chain["status"] == "converged")
    ]
    .pivot_table(
        index="maturity",
        columns="strike",
        values="implied_vol"
    )
    .sort_index()
)

plt.figure(figsize=(12, 6))
plt.imshow(
    call_surface.values,
    aspect="auto",
    origin="lower",
    extent=[
        call_surface.columns.min(),
        call_surface.columns.max(),
        call_surface.index.min(),
        call_surface.index.max()
    ]
)
plt.colorbar(label="Recovered call implied volatility")
plt.title("Recovered Call Implied Volatility Surface")
plt.xlabel("Strike")
plt.ylabel("Maturity")
plt.show()

## 18. Low-Vega failure cases

Implied volatility is hardest to infer when Vega is small.

We inspect solver diagnostics by Vega bucket.

This helps identify options where the IV number exists mathematically but is not economically reliable.

In [ ]:
converged = solved_chain[solved_chain["status"] == "converged"].copy()
converged["vega_bucket"] = pd.qcut(
    converged["vega"],
    q=5,
    labels=["lowest", "low", "middle", "high", "highest"]
)

vega_bucket_summary = (
    converged
    .groupby("vega_bucket", observed=True)
    .agg(
        n=("implied_vol", "count"),
        mean_vega=("vega", "mean"),
        mean_abs_iv_error=("abs_iv_error", "mean"),
        mean_iterations=("iterations", "mean"),
        max_abs_iv_error=("abs_iv_error", "max")
    )
    .reset_index()
)

vega_bucket_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    vega_bucket_summary["vega_bucket"].astype(str),
    vega_bucket_summary["mean_abs_iv_error"]
)
plt.title("Mean Absolute IV Error by Vega Bucket")
plt.xlabel("Vega bucket")
plt.ylabel("Mean absolute IV error")
plt.show()

## 19. Static no-arbitrage checks across strikes

For European calls at fixed maturity, call prices should be:

1. decreasing in strike;
2. convex in strike.

The decreasing condition is:

$$
C(K_1) \geq C(K_2)
\quad \text{for } K_1 < K_2
$$
Convexity means the second finite difference in strike should be non-negative.

These are simple checks. Real option surface arbitrage checking is more complex, especially with uneven strikes and bid/ask quotes.

In [ ]:
def static_arbitrage_checks_by_maturity(
    chain: pd.DataFrame,
    option_type: str = "call"
) -> pd.DataFrame:
    """
    Basic monotonicity and convexity checks across strikes for each maturity.
    """
    rows = []

    for maturity, group in chain[chain["option_type"] == option_type].groupby("maturity"):
        g = group.sort_values("strike").copy()

        strikes = g["strike"].to_numpy()
        prices = g["mid"].to_numpy()

        monotonic_violations = 0
        convexity_violations = 0

        if option_type == "call":
            monotonic_violations = int(np.sum(np.diff(prices) > 1e-8))
        elif option_type == "put":
            monotonic_violations = int(np.sum(np.diff(prices) < -1e-8))

        if len(prices) >= 3:
            # This simple check assumes equally spaced strikes.
            second_diff = prices[:-2] - 2 * prices[1:-1] + prices[2:]
            convexity_violations = int(np.sum(second_diff < -1e-6))

        rows.append({
            "maturity": maturity,
            "option_type": option_type,
            "n_quotes": len(g),
            "monotonicity_violations": monotonic_violations,
            "convexity_violations": convexity_violations
        })

    return pd.DataFrame(rows)


call_arb_checks = static_arbitrage_checks_by_maturity(option_chain_validated, option_type="call")
put_arb_checks = static_arbitrage_checks_by_maturity(option_chain_validated, option_type="put")

pd.concat([call_arb_checks, put_arb_checks], ignore_index=True).head(10)

## 20. Optional SciPy Brent check

If SciPy is installed, we can compare our bisection solver with SciPy's `brentq`.

Brent-style solvers combine bracketing safety with faster interpolation steps.

This section is optional so that the notebook remains self-contained.

In [ ]:
try:
    from scipy.optimize import brentq
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
def implied_vol_scipy_brentq(
    market_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    min_vol: float = 1e-8,
    max_vol: float = 5.0
) -> float | None:
    """
    Optional SciPy brentq implied volatility solver.
    """
    if not SCIPY_AVAILABLE:
        return None

    bounds = check_price_bounds(
        market_price=market_price,
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        option_type=option_type
    )

    if not bounds["within_bounds"]:
        return None

    def objective(vol):
        return (
            bsm_price(
                spot=spot,
                strike=strike,
                maturity=maturity,
                risk_free_rate=risk_free_rate,
                dividend_yield=dividend_yield,
                volatility=vol,
                option_type=option_type
            )
            - market_price
        )

    try:
        return float(brentq(objective, min_vol, max_vol, xtol=1e-12, rtol=1e-12, maxiter=100))
    except Exception:
        return None


if SCIPY_AVAILABLE:
    sample_row = option_chain_validated[
        option_chain_validated["quote_is_valid"]
    ].iloc[0]

    brent_iv = implied_vol_scipy_brentq(
        market_price=float(sample_row["mid"]),
        spot=float(sample_row["spot"]),
        strike=float(sample_row["strike"]),
        maturity=float(sample_row["maturity"]),
        risk_free_rate=float(sample_row["risk_free_rate"]),
        dividend_yield=float(sample_row["dividend_yield"]),
        option_type=str(sample_row["option_type"])
    )

    robust_iv = implied_vol_robust(
        market_price=float(sample_row["mid"]),
        spot=float(sample_row["spot"]),
        strike=float(sample_row["strike"]),
        maturity=float(sample_row["maturity"]),
        risk_free_rate=float(sample_row["risk_free_rate"]),
        dividend_yield=float(sample_row["dividend_yield"]),
        option_type=str(sample_row["option_type"])
    )[0].implied_vol

    display(pd.Series({
        "scipy_brentq_iv": brent_iv,
        "robust_solver_iv": robust_iv,
        "difference": None if brent_iv is None else brent_iv - robust_iv
    }))
else:
    print("SciPy unavailable; optional Brent check skipped.")

## 21. Solver status audit table

For an option chain, the implied volatility dataset should preserve solver status.

A clean downstream volatility surface should exclude or separately handle:

- failed bounds;
- crossed markets;
- low-Vega contracts;
- maximum-iteration failures;
- stale quotes.

Do not silently drop failures without recording why.

In [ ]:
audit_columns = [
    "strike",
    "maturity",
    "option_type",
    "bid",
    "ask",
    "mid",
    "quote_spread",
    "quote_is_valid",
    "quote_issues",
    "status",
    "method",
    "iterations",
    "implied_vol",
    "true_iv",
    "iv_error",
    "vega",
    "message",
    "bad_quote_type"
]

audit_table = solved_chain[audit_columns].copy()

audit_table.head()

In [ ]:
audit_summary = (
    audit_table
    .groupby(["quote_issues", "status"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

audit_summary

## 22. Saving implied volatility outputs

The notebook saves:

1. full option chain with bid/ask and IV solver diagnostics;
2. iteration logs;
3. quote validation summary;
4. arbitrage checks;
5. Vega bucket diagnostics;
6. call IV surface table;
7. manifest.

These files are useful for later volatility surface and option alpha notebooks.

In [ ]:
output_dir = Path("data/processed/implied_volatility_root_finding_v1")
output_dir.mkdir(parents=True, exist_ok=True)

solved_chain_path = output_dir / "synthetic_option_chain_with_implied_vols.csv"
iteration_log_path = output_dir / "solver_iteration_logs.csv"
audit_summary_path = output_dir / "solver_audit_summary.csv"
accuracy_summary_path = output_dir / "iv_accuracy_summary.csv"
vega_summary_path = output_dir / "vega_bucket_summary.csv"
call_arb_path = output_dir / "call_static_arbitrage_checks.csv"
put_arb_path = output_dir / "put_static_arbitrage_checks.csv"
call_surface_path = output_dir / "call_implied_vol_surface.csv"
manifest_path = output_dir / "manifest.json"

solved_chain.to_csv(solved_chain_path, index=False)
iteration_logs.to_csv(iteration_log_path, index=False)
audit_summary.to_csv(audit_summary_path, index=False)
accuracy_summary.to_csv(accuracy_summary_path, index=False)
vega_bucket_summary.to_csv(vega_summary_path, index=False)
call_arb_checks.to_csv(call_arb_path, index=False)
put_arb_checks.to_csv(put_arb_path, index=False)
call_surface.to_csv(call_surface_path)

manifest = {
    "dataset_name": "synthetic_implied_volatility_root_finding",
    "schema_version": "implied_volatility_root_finding_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_quotes": int(len(solved_chain)),
    "n_converged": int((solved_chain["status"] == "converged").sum()),
    "n_failed_bounds": int((solved_chain["status"] == "failed_bounds").sum()),
    "n_bad_quotes_injected": int(solved_chain["is_bad_quote_injected"].sum()),
    "methods": sorted(solved_chain["method"].dropna().unique().tolist()),
    "notes": [
        "Implied volatility is solved from BSM prices using Newton with bisection fallback.",
        "Quotes outside theoretical bounds are flagged and not inverted.",
        "Low-Vega options may have unstable implied volatilities.",
        "Synthetic true IV is included only because this is a controlled simulation."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

solved_chain_path, iteration_log_path, audit_summary_path, manifest_path

## 23. Practical checklist for IV extraction

Before trusting implied volatility data, check:

1. **Price bounds**  
   Is the quote inside theoretical no-arbitrage bounds?

2. **Bid/ask sanity**  
   Is the market crossed? Is bid non-negative? Is ask above bid?

3. **Option specification**  
   Are strike, maturity, option type, multiplier, expiry, and underlying correct?

4. **Rates and dividends**  
   Are the correct discount and forward inputs used?

5. **Exercise style**  
   BSM European IV may not be appropriate for American options with early exercise.

6. **Vega**  
   Is Vega large enough for the implied volatility to be numerically meaningful?

7. **Solver status**  
   Did the solver converge? Which method was used?

8. **Surface consistency**  
   Are prices monotone and convex across strikes?

9. **Outlier handling**  
   Are stale quotes, zero bids, and illiquid contracts flagged?

10. **Audit trail**  
   Can every missing or failed IV be explained?

## 24. Limitations

This notebook deliberately uses a simplified BSM inversion setup.

### 24.1 European option assumption

The BSM formula prices European options.

Many listed equity options are American-style, which can make BSM implied volatility an approximation rather than a perfect model inversion.

### 24.2 Synthetic quotes are cleaner than market quotes

Real option chains can contain:

- stale quotes;
- crossed markets;
- zero bids;
- wide spreads;
- missing maturities;
- corporate actions;
- early exercise effects;
- settlement convention issues.

### 24.3 Rates and dividends are simplified

The notebook uses constant rates and dividend yield.

Real surfaces may require:

- discount curves;
- borrow curves;
- dividend forecasts;
- futures forwards;
- collateral rates;
- funding spreads.

### 24.4 Low-Vega instability

Some options have mathematically defined implied volatility but very unstable numerical inversion.

A tiny price change can create a huge IV change.

### 24.5 Static arbitrage checks are basic

The notebook checks simple monotonicity and convexity across strikes.

A full surface validation would also check calendar arbitrage, butterfly arbitrage, bid/ask feasibility, and arbitrage-free smoothing.

### 24.6 BSM IV is a coordinate system, not the truth

A volatility smile is evidence that the BSM constant-volatility assumption is not literally correct.

Implied volatility is best interpreted as a market quotation convention and model coordinate, not a direct physical forecast.

## 25. Research frontier and current directions

Implied volatility extraction is classical, but still important in modern option systems because option chains can contain thousands of contracts across strikes, maturities, and underlyings.

### 25.1 Fast and robust IV algorithms

Naïve Newton-Raphson can fail for extreme strikes, short maturities, or prices close to intrinsic value.

Modern solvers use:

- carefully chosen initial guesses;
- analytical bounds;
- transformed variables such as total variance;
- rational approximations;
- hybrid Newton/bracketing methods.

A future notebook could implement Peter Jäckel-style rational approximations and compare them with Newton and bisection.

### 25.2 Batched and GPU-accelerated IV extraction

Market makers and volatility desks often need to invert large option chains quickly.

Modern workflows can batch IV calculations across thousands or millions of contracts using:

- vectorised NumPy;
- Numba;
- JAX;
- PyTorch;
- CUDA/Triton;
- C++ kernels.

A future notebook could benchmark batched implied-volatility extraction across CPU and GPU backends.

### 25.3 Arbitrage-free volatility surface fitting

Raw implied volatilities are noisy.

A production volatility surface should usually be smoothed while preserving no-arbitrage constraints.

Common approaches include:

- SVI;
- SABR;
- local volatility;
- Gaussian process smoothing;
- neural surface models with arbitrage penalties.

A future notebook could fit an SVI smile and test for butterfly arbitrage.

### 25.4 Implied volatility under alternative models

BSM IV is only one coordinate system.

Other markets use:

- Black-76 implied volatility for futures options;
- normal/Bachelier volatility for rates;
- displaced diffusion;
- local volatility;
- stochastic volatility;
- jump-diffusion.

A future notebook could compare BSM IV with Black-76 IV for futures options.

### 25.5 IV as a signal

Implied volatility surfaces can be used for research signals:

- skew steepness;
- term structure;
- volatility risk premium;
- realised-versus-implied spread;
- smile curvature;
- event volatility.

A future notebook could turn the recovered IV surface into predictive features for volatility and directional alpha.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_09_volatility_surface_construction**  
   Building and validating an implied volatility surface.

2. **02_11_heston_model_calibration**  
   Calibrating a stochastic volatility model to implied volatilities.

3. **02_12_sabr_smile_calibration**  
   Fitting a parametric smile model.

4. **02_14_local_volatility_dupire_surface**  
   Using implied volatility surfaces to infer local volatility.

5. **03_09_volatility_surface_alpha_signals**  
   Extracting predictive features from skew and term structure.

6. **04_07_beta_weighted_tail_risk_hedging**  
   Using puts and implied volatility for crash protection.

7. **06_11_cpp_python_bindings_pybind11**  
   Moving a batched IV solver into compiled code.

## 27. Summary

This notebook built a robust implied-volatility root-finding pipeline.

It showed how to:

1. define implied volatility as a BSM root;
2. check theoretical price bounds;
3. use Vega to run Newton-Raphson;
4. fall back to bisection when Newton fails;
5. process a full option chain;
6. reject bad quotes;
7. recover a synthetic volatility smile and surface;
8. diagnose low-Vega instability;
9. check simple static arbitrage conditions;
10. save an auditable IV dataset.

The key computational takeaway is:

> An implied-volatility solver should return diagnostics, not just a number.

The key financial takeaway is:

> Implied volatility is a model-implied coordinate extracted from option prices. Its reliability depends on quote quality, model assumptions, and numerical stability.

## 28. Further reading

### Classical foundations

- Black, F. and Scholes, M. "The Pricing of Options and Corporate Liabilities."
- Merton, R. C. "Theory of Rational Option Pricing."
- Hull, J. C. *Options, Futures, and Other Derivatives.*

### Numerical root finding

- Newton-Raphson method.
- Bisection method.
- Brent's method.
- SciPy `optimize.newton`.
- SciPy `optimize.brentq`.

### Implied volatility algorithms

- Jäckel, P. *Let's Be Rational.*
- Tehranchi, M. R. "Uniform bounds for Black-Scholes implied volatility."
- Choi, Huh, and Su on tighter implied-volatility bounds and root finding.
- Modern batched implied-volatility solvers using JAX, CUDA, and compiled kernels.

### Future extensions

- SVI smile fitting.
- SABR calibration.
- Black-76 futures option IV.
- Bachelier normal volatility.
- Arbitrage-free surface smoothing.
- Local volatility and Dupire inversion.